In [7]:
# ============================================================
# V4 — SOURCE TARGET ENCODING + TWO-STAGE FUSION
# CV-safe, leaderboard-consistent
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix


# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X = df.drop(columns=["label"])
y = df["label"].values
N_CLASSES = len(np.unique(y))


# ============================================================
# (A) SOURCE TARGET ENCODING (SOFT LEAKAGE, CV-SAFE)
# ============================================================
class SourceTargetEncoder(BaseEstimator, TransformerMixin):
	def __init__(self, n_classes, smoothing=10):
		self.n_classes = n_classes
		self.smoothing = smoothing

	def fit(self, X, y):
		df = pd.DataFrame({
			"source": X["source"].values,
			"label": y
		})

		self.global_prior_ = np.bincount(y, minlength=self.n_classes) / len(y)

		stats = (
			df.groupby("source")["label"]
			.value_counts()
			.unstack(fill_value=0)
		)

		self.source_te_ = {}
		for src, row in stats.iterrows():
			counts = row.values
			total = counts.sum()

			te = (counts + self.smoothing * self.global_prior_) / \
			     (total + self.smoothing)

			self.source_te_[src] = te

		return self

	def transform(self, X):
		out = np.zeros((len(X), self.n_classes))
		for i, src in enumerate(X["source"].values):
			out[i] = self.source_te_.get(src, self.global_prior_)
		return out


# ============================================================
# TEXT VECTORIZERS (SAME AS STRATEGY 2)
# ============================================================
ARTICLE_TFIDF = TfidfVectorizer(
	max_features=120_000,
	ngram_range=(1,2),
	min_df=3,
	max_df=0.9,
	sublinear_tf=True,
	stop_words="english"
)

TITLE_TFIDF = TfidfVectorizer(
	max_features=30_000,
	ngram_range=(1,2),
	min_df=2,
	max_df=0.95,
	sublinear_tf=True,
	stop_words="english"
)


# ============================================================
# (B1) PIPELINE — TEXT + SOURCE TARGET ENCODING
# ============================================================
def make_text_plus_source_te(C=1.0):
	prep = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
			("source_te", SourceTargetEncoder(N_CLASSES), ["source"]),
		],
		n_jobs=-1
	)

	return Pipeline([
		("prep", prep),
		("clf", LogisticRegression(
			C=C,
			max_iter=2000,
			n_jobs=-1,
			multi_class="ovr"
		))
	])


# ============================================================
# (B2) TWO-STAGE MODEL
# ============================================================
def make_source_only_model():
	return Pipeline([
		("ohe", OneHotEncoder(handle_unknown="ignore")),
		("clf", LogisticRegression(
			C=5.0,
			max_iter=2000,
			n_jobs=-1,
			multi_class="ovr"
		))
	])

def make_text_only_model():
	prep = ColumnTransformer(
		transformers=[
			("article", ARTICLE_TFIDF, "article"),
			("title", TITLE_TFIDF, "title"),
		],
		n_jobs=-1
	)

	return Pipeline([
		("prep", prep),
		("clf", LogisticRegression(
			C=1.0,
			max_iter=2000,
			n_jobs=-1,
			multi_class="ovr"
		))
	])

def fuse_probs(p_text, p_source, alpha=0.7):
	return alpha * p_text + (1 - alpha) * p_source


# ============================================================
# CV EVALUATION
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n==============================")
print("MODEL A — TEXT + SOURCE TE")
print("==============================")

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	Xtr, Xte = X.iloc[tr], X.iloc[te]
	ytr, yte = y[tr], y[te]

	model = make_text_plus_source_te(C=1.0)
	model.fit(Xtr, ytr)

	yp = model.predict(Xte)

	f1s.append(f1_score(yte, yp, average="macro"))
	recalls.append(recall_score(yte, yp, average="macro"))
	cms.append(confusion_matrix(yte, yp))

print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


print("\n==============================")
print("MODEL B — TWO-STAGE FUSION")
print("==============================")

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	Xtr, Xte = X.iloc[tr], X.iloc[te]
	ytr, yte = y[tr], y[te]

	src_model = make_source_only_model()
	txt_model = make_text_only_model()

	src_model.fit(Xtr[["source"]], ytr)
	txt_model.fit(Xtr, ytr)

	p_src = src_model.predict_proba(Xte[["source"]])
	p_txt = txt_model.predict_proba(Xte)

	p = fuse_probs(p_txt, p_src, alpha=0.7)
	yp = p.argmax(axis=1)

	f1s.append(f1_score(yte, yp, average="macro"))
	recalls.append(recall_score(yte, yp, average="macro"))
	cms.append(confusion_matrix(yte, yp))

print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))



MODEL A — TEXT + SOURCE TE


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\msist\AppData\Roaming\Python\Python311\site

Macro F1: 0.6979066470545858
Macro Recall: 0.695167022134395
Confusion Matrix:
 [[19445   696   455   750   249  1712   234]
 [  779  8284   637   354    66   345   123]
 [  728   594  9148   328    51   211   101]
 [ 1902   591   569  4950   642  1099   224]
 [  291    57    14   280  7648   277     7]
 [ 4320   658   324  1064   665  5740   282]
 [  513   173    69   218    34   202  1893]]

MODEL B — TWO-STAGE FUSION


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\msist\AppData\Roaming\Python\Python311\site

Macro F1: 0.6822438096363932
Macro Recall: 0.6745205171652962
Confusion Matrix:
 [[19828   664   443   512   315  1577   202]
 [  925  8099   822   233    43   361   105]
 [ 1019   504  9088   243    44   165    98]
 [ 2507   555   590  4439   639  1052   195]
 [  443    40    35   189  7596   265     6]
 [ 4639   664   506   850   688  5436   270]
 [  686   157   119   148    27   250  1715]]


In [8]:
import pandas as pd

# ============================================================
# LOAD EVAL
# ============================================================
eval_df = pd.read_csv("../data/raw/evaluation.csv")  # cambia nome se serve

print("Shape eval:", eval_df.shape)
print("Columns:", eval_df.columns.tolist())

# ============================================================
# CHECK TIMESTAMP PRESENCE
# ============================================================
assert "timestamp" in eval_df.columns, "❌ Colonna 'timestamp' non trovata"

ts_missing = eval_df["timestamp"].isna()

n_total = len(eval_df)
n_missing = ts_missing.sum()
n_present = n_total - n_missing

print("\n=== TIMESTAMP AVAILABILITY (EVAL) ===")
print(f"Total samples:      {n_total}")
print(f"Timestamp present:  {n_present} ({n_present / n_total:.2%})")
print(f"Timestamp missing:  {n_missing} ({n_missing / n_total:.2%})")

# ============================================================
# OPTIONAL: PARSE TIMESTAMP IF PRESENT
# ============================================================
if n_present > 0:
	eval_df["timestamp_parsed"] = pd.to_datetime(
		eval_df["timestamp"],
		errors="coerce"
	)

	print("\nParsed timestamp stats:")
	print(eval_df["timestamp_parsed"].describe())

	# Quick derived features (for sanity check)
	eval_df["year"] = eval_df["timestamp_parsed"].dt.year
	eval_df["month"] = eval_df["timestamp_parsed"].dt.month
	eval_df["hour"] = eval_df["timestamp_parsed"].dt.hour

	print("\nYear distribution:")
	print(eval_df["year"].value_counts(dropna=False).sort_index())

	print("\nMonth distribution:")
	print(eval_df["month"].value_counts(dropna=False).sort_index())


Shape eval: (20000, 6)
Columns: ['Id', 'source', 'title', 'article', 'page_rank', 'timestamp']

=== TIMESTAMP AVAILABILITY (EVAL) ===
Total samples:      20000
Timestamp present:  20000 (100.00%)
Timestamp missing:  0 (0.00%)

Parsed timestamp stats:
count                            13019
mean     2006-10-22 21:40:16.657270016
min                2004-08-18 23:00:26
25%         2006-07-06 19:25:29.500000
50%                2007-02-28 02:30:40
75%                2007-09-18 15:51:27
max                2008-02-20 22:54:57
Name: timestamp_parsed, dtype: object

Year distribution:
year
2004.0    2826
2005.0     349
2006.0    2398
2007.0    5619
2008.0    1827
NaN       6981
Name: count, dtype: int64

Month distribution:
month
1.0     1446
2.0     1702
3.0      440
5.0      321
6.0     1043
7.0      750
8.0      767
9.0     2276
10.0    1426
11.0     875
12.0    1973
NaN     6981
Name: count, dtype: int64


In [10]:
import pandas as pd

# ============================================================
# LOAD DATA
# ============================================================
dev_df  = pd.read_csv("../data/raw/development.csv")
eval_df = pd.read_csv("../data/raw/evaluation.csv")

print("DEV shape:", dev_df.shape)
print("EVAL shape:", eval_df.shape)

# ============================================================
# SAFETY CHECKS
# ============================================================
assert "Id" in dev_df.columns, "❌ 'Id' missing in dev"
assert "Id" in eval_df.columns, "❌ 'Id' missing in eval"
assert "timestamp" in dev_df.columns, "❌ 'timestamp' missing in dev"
assert "timestamp" in eval_df.columns, "❌ 'timestamp' missing in eval"

# ============================================================
# PARSE TIMESTAMPS
# ============================================================
dev_df["ts_dev"]  = pd.to_datetime(dev_df["timestamp"],  errors="coerce")
eval_df["ts_eval"] = pd.to_datetime(eval_df["timestamp"], errors="coerce")

dev_df["has_ts_dev"]   = dev_df["ts_dev"].notna()
eval_df["has_ts_eval"] = eval_df["ts_eval"].notna()

# ============================================================
# MERGE DEV ↔ EVAL ON Id
# ============================================================
merged = dev_df[["Id", "has_ts_dev"]].merge(
	eval_df[["Id", "has_ts_eval"]],
	on="Id",
	how="inner"
)

print("\nMerged shape:", merged.shape)

# ============================================================
# CORE ANALYSIS
# ============================================================
# 1) DEV missing → EVAL present
case_1 = merged[
	(merged["has_ts_dev"] == False) &
	(merged["has_ts_eval"] == True)
]

# 2) DEV present → EVAL present
case_2 = merged[
	(merged["has_ts_dev"] == True) &
	(merged["has_ts_eval"] == True)
]

# 3) DEV missing → EVAL missing (should be 0)
case_3 = merged[
	(merged["has_ts_dev"] == False) &
	(merged["has_ts_eval"] == False)
]

# ============================================================
# REPORT
# ============================================================
n_total = len(merged)

print("\n=== DEV → EVAL TIMESTAMP TRANSITION ===")
print(f"Total matched samples: {n_total}")

print(f"\nDEV missing → EVAL present:")
print(f"Count: {len(case_1)} ({len(case_1)/n_total:.2%})")

print(f"\nDEV present → EVAL present:")
print(f"Count: {len(case_2)} ({len(case_2)/n_total:.2%})")

print(f"\nDEV missing → EVAL missing:")
print(f"Count: {len(case_3)} ({len(case_3)/n_total:.2%})")

# ============================================================
# OPTIONAL: SAVE IDS FOR FURTHER ANALYSIS
# ============================================================
case_1.to_csv("dev_missing_eval_present_ids.csv", index=False)
print("\nSaved: dev_missing_eval_present_ids.csv")


DEV shape: (79997, 7)
EVAL shape: (20000, 6)

Merged shape: (20000, 3)

=== DEV → EVAL TIMESTAMP TRANSITION ===
Total matched samples: 20000

DEV missing → EVAL present:
Count: 4541 (22.71%)

DEV present → EVAL present:
Count: 8478 (42.39%)

DEV missing → EVAL missing:
Count: 2405 (12.03%)

Saved: dev_missing_eval_present_ids.csv


In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix


# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv("../data/raw/development.csv")

# safety
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

# timestamp parse
df["timestamp_parsed"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["has_timestamp"] = df["timestamp_parsed"].notna().astype(int)

# ============================================================
# COARSE TIME FEATURES (robust)
# ============================================================
df["year"]   = df["timestamp_parsed"].dt.year
df["month"]  = df["timestamp_parsed"].dt.month
df["dow"]    = df["timestamp_parsed"].dt.dayofweek
df["hour"]   = df["timestamp_parsed"].dt.hour

# hour bucket (0..5)
# night(0-5), morning(6-10), late_morning(11-13), afternoon(14-17), evening(18-21), late(22-23)
def hour_bucket(h):
	if pd.isna(h): return np.nan
	h = int(h)
	if 0 <= h <= 5: return 0
	if 6 <= h <= 10: return 1
	if 11 <= h <= 13: return 2
	if 14 <= h <= 17: return 3
	if 18 <= h <= 21: return 4
	return 5

df["hour_bucket"] = df["hour"].apply(hour_bucket)

# ============================================================
# IMPUTATION (DO NOT "PREDICT" TIMESTAMP)
# Use medians + has_timestamp mask
# ============================================================
for col in ["year","month","dow","hour_bucket"]:
	med = df[col].median(skipna=True)
	df[col] = df[col].fillna(med)

# labels
X = df.drop(columns=["label"])
y = df["label"]

# baseline numeric if exist
NUM_COLS = [c for c in ["title_ratio", "n_tokens"] if c in X.columns]

TIME_COLS = ["has_timestamp","year","month","dow","hour_bucket"]


# ============================================================
# PREPROCESSOR
# ============================================================
preprocess = ColumnTransformer(
	transformers=[
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),

		("num", StandardScaler(), NUM_COLS),

		("time", StandardScaler(), TIME_COLS),
	],
	n_jobs=-1
)

model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=2000,
		n_jobs=-1
	))
])


# ============================================================
# CV
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("\nSTRATEGY 2 + TIMESTAMP (masked/imputed + has_timestamp)")
print("Macro F1:", float(np.mean(f1s)))
print("Macro Recall:", float(np.mean(recalls)))
print("Confusion Matrix:\n", np.sum(cms, axis=0))



STRATEGY 2 + TIMESTAMP (masked/imputed + has_timestamp)
Macro F1: 0.7001935407919392
Macro Recall: 0.6966032226875332
Confusion Matrix:
 [[18932   644   388   765   198  2382   233]
 [  755  8310   552   343    73   446   109]
 [  684   616  9065   369    46   269   112]
 [ 1696   568   519  4969   632  1380   213]
 [  243    62    17   320  7581   347     4]
 [ 3799   626   299  1162   641  6256   270]
 [  423   147    97   212    33   287  1903]]
